## Setup, Load Features & Initialize CLIP

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

PHASE2_ROOT = Path.cwd().resolve().parent
if str(PHASE2_ROOT) not in sys.path:
    sys.path.insert(0, str(PHASE2_ROOT))

from src.clip_eval import (
    DEFAULT_MODEL_NAME,
    PROMPT_LADDER,
    build_genre_text_embeddings,
    encode_images,
    load_clip,
    score_embeddings,
)
from src.paths import (
    FEATURES_PARQUET,
    PREDICTIONS_PARQUET,
    SELECTED_6_GENRES,
    ensure_dirs,
)

ensure_dirs()

df_features = pd.read_parquet(FEATURES_PARQUET)
print(f"Loaded {len(df_features):,} covers from {FEATURES_PARQUET}")
print(df_features["benchmark"].value_counts())

model, processor, device = load_clip(DEFAULT_MODEL_NAME)
logit_scale = float(model.logit_scale.exp().item())
print(f"Loaded {DEFAULT_MODEL_NAME} on {device} (logit_scale={logit_scale:.2f})")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loaded 6,000 covers from /Users/darina/Desktop/JKU_AI_Masterr/YEAR 2/AI & VISUALIZATION/MarshmallowBunnies/phase2-ai-assisted/outputs/visual_features.parquet
benchmark
6genre_p1_exact    3000
30genre_test       3000
Name: count, dtype: int64
Loaded openai/clip-vit-base-patch32 on mps (logit_scale=100.00)


## Encode Original & Text-Masked Cover Embeddings

In [2]:
orig_paths = [Path(p) for p in df_features["image_path"]]
mask_paths = [Path(p) for p in df_features["masked_image_path"]]

print("Encoding original covers...")
orig_embeddings = encode_images(model, processor, orig_paths, device=device, batch_size=64)

print("Encoding text-masked covers...")
mask_embeddings = encode_images(model, processor, mask_paths, device=device, batch_size=64)

print("Embedding tensor shapes:", orig_embeddings.shape, mask_embeddings.shape)

Encoding original covers...
Encoding text-masked covers...
Embedding tensor shapes: torch.Size([6000, 512]) torch.Size([6000, 512])


## Evaluate All 4 Prompt-Ladder Levels Across Both Benchmarks

In [3]:
benchmark_configs = {
    "6genre_p1_exact": SELECTED_6_GENRES,
    "30genre_test": sorted(
        df_features.loc[df_features["benchmark"] == "30genre_test", "genre"].unique().tolist()
    ),
}

scored_parts = []

for bench_name, genres in benchmark_configs.items():
    mask_idx = df_features.index[df_features["benchmark"] == bench_name].to_numpy()
    df_sub = df_features.loc[mask_idx].reset_index(drop=True)
    orig_sub = orig_embeddings[mask_idx]
    mask_sub = mask_embeddings[mask_idx]

    level_frames = [df_sub]

    for level_name, templates in PROMPT_LADDER.items():
        text_emb = build_genre_text_embeddings(
            model=model,
            processor=processor,
            genres=genres,
            templates=templates,
            device=device,
        )

        df_orig_scores = score_embeddings(
            image_embeddings=orig_sub,
            text_embeddings=text_emb,
            genres=genres,
            logit_scale=logit_scale,
            prefix=f"orig_{level_name}",
        )
        df_mask_scores = score_embeddings(
            image_embeddings=mask_sub,
            text_embeddings=text_emb,
            genres=genres,
            logit_scale=logit_scale,
            prefix=f"mask_{level_name}",
        )
        level_frames.extend([df_orig_scores, df_mask_scores])

    scored_parts.append(pd.concat(level_frames, axis=1))

df_predictions = pd.concat(scored_parts, ignore_index=True)
df_predictions.to_parquet(PREDICTIONS_PARQUET, index=False)
print(f"Saved {len(df_predictions):,} scored rows to {PREDICTIONS_PARQUET}")

Saved 6,000 scored rows to /Users/darina/Desktop/JKU_AI_Masterr/YEAR 2/AI & VISUALIZATION/MarshmallowBunnies/phase2-ai-assisted/outputs/phase2_predictions.parquet


## Summary Table: Prompt Ladder, OCR Leakage & Flip Stability

In [4]:
summary_rows = []

for bench_name in ["6genre_p1_exact", "30genre_test"]:
    sub = df_predictions[df_predictions["benchmark"] == bench_name]
    neutral_orig_pred = sub["orig_L2_neutral_pred"]

    for level_name in PROMPT_LADDER.keys():
        orig_pred = sub[f"orig_{level_name}_pred"]
        mask_pred = sub[f"mask_{level_name}_pred"]

        orig_acc = (orig_pred == sub["genre"]).mean()
        mask_acc = (mask_pred == sub["genre"]).mean()
        ocr_drop = orig_acc - mask_acc
        ocr_flip_rate = (orig_pred != mask_pred).mean()
        prompt_flip_rate = (orig_pred != neutral_orig_pred).mean()
        mean_margin = sub[f"orig_{level_name}_margin"].mean()

        summary_rows.append(
            {
                "Benchmark": bench_name,
                "Prompt Level": level_name,
                "Orig Acc (%)": round(orig_acc * 100, 2),
                "Masked Acc (%)": round(mask_acc * 100, 2),
                "OCR Acc Drop (pp)": round(ocr_drop * 100, 2),
                "OCR Flip Rate (%)": round(ocr_flip_rate * 100, 2),
                "Flips vs L2_neutral (%)": round(prompt_flip_rate * 100, 2),
                "Mean Margin": round(mean_margin, 4),
            }
        )

df_summary = pd.DataFrame(summary_rows)
display(df_summary)

,Benchmark,Prompt Level,Orig Acc (%),Masked Acc (%),OCR Acc Drop (pp),OCR Flip Rate (%),Flips vs L2_neutral (%),Mean Margin
0,6genre_p1_exact,L1_minimal,41.67,39.80,1.87,27.10,38.07,0.0185
1,6genre_p1_exact,L2_neutral,61.10,53.73,7.37,25.70,0.00,0.0152
2,6genre_p1_exact,L3_stylistic_mood,60.47,53.20,7.27,23.97,19.83,0.0139
3,6genre_p1_exact,L4_prompt_ensemble,61.17,57.67,3.50,24.43,24.33,0.0153
4,30genre_test,L1_minimal,25.40,19.57,5.83,46.80,62.50,0.0153
5,30genre_test,L2_neutral,32.27,25.00,7.27,47.90,0.00,0.0121
6,30genre_test,L3_stylistic_mood,33.00,23.93,9.07,44.17,47.97,0.0118
7,30genre_test,L4_prompt_ensemble,33.93,25.90,8.03,46.70,35.30,0.0112
